In [ ]:
# config.py
import torch
import os
import matplotlib.pyplot as plt
from PIL import Image

# --- Training Hyperparameters ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LEARNING_RATE_GEN = 1e-4
LEARNING_RATE_DISC = 1e-4
BATCH_SIZE = 16
NUM_EPOCHS_PRETRAIN = 50
NUM_EPOCHS_GAN = 100
HIGH_RES_SIZE = 96  # As per the paper
LOW_RES_SIZE = HIGH_RES_SIZE // 4
NUM_WORKERS = 4
LAMBDA_VGG = 1.0  # Weight for VGG/content loss
LAMBDA_ADV = 1e-3  # Weight for adversarial loss

os.makedirs("saved_models", exist_ok=True)

# --- Model Paths ---
PRETRAINED_GEN_PATH = "saved_models/srresnet_pretrained.pth"
GEN_PATH = "saved_models/generator.pth"
DISC_PATH = "saved_models/discriminator.pth"

# --- Dataset Paths ---
TRAIN_DIR = "/root/.cache/kagglehub/datasets/takihasan/div2k-dataset-for-super-resolution/versions/1/Dataset/DIV2K_train_HR"
TEST_DIR = "/root/.cache/kagglehub/datasets/takihasan/div2k-dataset-for-super-resolution/versions/1/Dataset/DIV2K_valid_HR"

# set the repo name
model_name = "keanteng/srgan-div2k-0723-v2"

In [ ]:
# dataset.py
import os
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms

class ImageDataset(Dataset):
    """
    Custom dataset to load high-resolution images and create low-resolution counterparts.
    """
    def __init__(self, hr_dir, hr_size):
        super(ImageDataset, self).__init__()
        self.hr_image_files = [os.path.join(hr_dir, f) for f in os.listdir(hr_dir)]
        self.hr_size = hr_size

        # Transform for the original image before cropping
        self.initial_transform = transforms.Compose([
            transforms.ToTensor(),
        ])

        # Normalization transforms
        self.hr_normalize = transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]) # normalize to [-1, 1]
        self.lr_normalize = transforms.Normalize(mean=[0.0, 0.0, 0.0], std=[1.0, 1.0, 1.0]) # nothing change x = (x - mean) / std, so if mean=0 and std=1, x remains unchanged, we will use tanh to scale to [-1,1] in the generator

    def __getitem__(self, index):
        # Load image
        hr_image = Image.open(self.hr_image_files[index]).convert("RGB")

        # Convert to tensor first
        hr_tensor = self.initial_transform(hr_image)

        # Apply random crop to get consistent size
        crop_transform = transforms.RandomCrop(self.hr_size)
        hr_cropped = crop_transform(hr_tensor)

        # Create LR version by downsampling the cropped HR image
        lr_tensor = transforms.functional.resize(
            hr_cropped,
            size=self.hr_size // 4,
            interpolation=transforms.InterpolationMode.BICUBIC
        )

        # Apply normalization
        hr_normalized = self.hr_normalize(hr_cropped)
        lr_normalized = self.lr_normalize(lr_tensor)

        return lr_normalized, hr_normalized

    def __len__(self):
        return len(self.hr_image_files)

In [ ]:
# loss.py
import torch
from torch import nn
from torchvision.models import vgg19

class VGGContentLoss(nn.Module):
    """
    Calculates the content loss in the VGG19 feature space.
    The paper uses the features from the layer before the 5th max-pooling layer (VGG54).
    In PyTorch's VGG19 implementation, this corresponds to `features[35]`.
    """
    def __init__(self, device):
        super(VGGContentLoss, self).__init__()
        vgg_model = vgg19(weights="DEFAULT").features[:36].to(device).eval()
        for param in vgg_model.parameters():
            param.requires_grad = False
        self.vgg_model = vgg_model
        self.loss = nn.MSELoss()

    def forward(self, generated, target):
        gen_features = self.vgg_model(generated)
        target_features = self.vgg_model(target)
        return self.loss(gen_features, target_features)

class PerceptualLoss(nn.Module):
    """
    Combined Perceptual Loss for SRGAN training.
    It includes VGG content loss and adversarial loss.
    """
    def __init__(self, device, lambda_vgg, lambda_adv):
        super(PerceptualLoss, self).__init__()
        self.vgg_loss_fn = VGGContentLoss(device)
        self.adversarial_loss_fn = nn.BCEWithLogitsLoss()
        self.lambda_vgg = lambda_vgg
        self.lambda_adv = lambda_adv

    def forward(self, disc_fake_output, gen_hr, hr_img):
        # Content Loss
        vgg_loss = self.vgg_loss_fn(gen_hr, hr_img)

        # Adversarial Loss (Generator's perspective)
        # We want the generator to fool the discriminator, so we compare its output to a tensor of ones.
        adversarial_loss = self.adversarial_loss_fn(disc_fake_output, torch.ones_like(disc_fake_output))

        # Total Perceptual Loss
        total_loss = self.lambda_vgg * vgg_loss + self.lambda_adv * adversarial_loss
        return total_loss

In [ ]:
# models.py
import torch
from torch import nn

class ResidualBlock(nn.Module):
    """
    A single residual block as defined in the SRGAN paper.
    It contains two convolutional layers with batch normalization and PReLU activation.
    """
    def __init__(self, in_channels):
        super(ResidualBlock, self).__init__()
        self.conv_block1 = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(in_channels),
            nn.PReLU(),
        )
        self.conv_block2 = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(in_channels),
        )

    def forward(self, x):
        identity = x
        out = self.conv_block1(x)
        out = self.conv_block2(out)
        return identity + out

class UpsampleBlock(nn.Module):
    """
    Upsampling block using a convolutional layer and PixelShuffle.
    This increases the resolution by a factor of 2.
    """
    def __init__(self, in_channels, scale_factor=2):
        super(UpsampleBlock, self).__init__()
        self.conv = nn.Conv2d(in_channels, in_channels * (scale_factor ** 2), kernel_size=3, stride=1, padding=1)
        self.pixel_shuffle = nn.PixelShuffle(scale_factor)
        self.prelu = nn.PReLU()

    def forward(self, x):
        return self.prelu(self.pixel_shuffle(self.conv(x)))

class Generator(nn.Module):
    """
    The Generator Network (SRResNet).
    It takes a low-resolution image and outputs a super-resolved version.
    """
    def __init__(self, in_channels=3, num_res_blocks=16):
        super(Generator, self).__init__()
        self.initial_conv = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=9, stride=1, padding=4),
            nn.PReLU()
        )

        self.residuals = nn.Sequential(*[ResidualBlock(64) for _ in range(num_res_blocks)])

        self.mid_conv = nn.Sequential(
            nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64)
        )

        # Upsampling by 4x (two 2x upsample blocks)
        self.upsample_blocks = nn.Sequential(
            UpsampleBlock(64),
            UpsampleBlock(64),
        )

        self.final_conv = nn.Conv2d(64, in_channels, kernel_size=9, stride=1, padding=4)

    def forward(self, x):
        initial_out = self.initial_conv(x)
        residual_out = self.residuals(initial_out)
        mid_out = self.mid_conv(residual_out)
        mid_out = mid_out + initial_out # Skip connection
        upsampled_out = self.upsample_blocks(mid_out)
        final_out = self.final_conv(upsampled_out)
        return torch.tanh(final_out) # Tanh activation to scale output to [-1, 1]

class Discriminator(nn.Module):
    """
    The Discriminator Network.
    It takes an image and outputs a probability of it being a real high-resolution image.
    """
    def __init__(self, in_channels=3):
        super(Discriminator, self).__init__()

        def conv_block(in_feat, out_feat, stride=1):
            return nn.Sequential(
                nn.Conv2d(in_feat, out_feat, kernel_size=3, stride=stride, padding=1),
                nn.BatchNorm2d(out_feat),
                nn.LeakyReLU(0.2, inplace=True)
            )

        self.blocks = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=3, stride=1, padding=1),
            nn.LeakyReLU(0.2, inplace=True),

            conv_block(64, 64, stride=2),
            conv_block(64, 128, stride=1),
            conv_block(128, 128, stride=2),
            conv_block(128, 256, stride=1),
            conv_block(256, 256, stride=2),
            conv_block(256, 512, stride=1),
            conv_block(512, 512, stride=2),
        )

        # The paper mentions flattening and then two dense layers
        # The output size after convolutions on a 96x96 image is 512x6x6
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), # Flattens the output
            nn.Conv2d(512, 1024, kernel_size=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(1024, 1, kernel_size=1)
        )

    def forward(self, x):
        batch_size = x.size(0)
        out = self.blocks(x)
        out = self.classifier(out)
        return out.view(batch_size, -1) # No sigmoid here, handled by BCEWithLogitsLoss

## From The Paper

In [ ]:
# train_gan.py
import torch
from torch import optim, nn
from torch.utils.data import DataLoader
from tqdm import tqdm
#import config
#from models import Generator, Discriminator
#from dataset import ImageDataset
#from loss import PerceptualLoss

def train_gan():

    dataset = ImageDataset(hr_dir=TRAIN_DIR, hr_size=HIGH_RES_SIZE)
    # load the dataset
    loader = DataLoader(
        dataset, # from the dataset class
        batch_size=BATCH_SIZE, # if 2 then 2 images will be loaded at once
        shuffle=True,
        # shuffle at once or batch by batch?
        # shuffle everything first say 1,2,3,4,5,6,7,8,9,10
        # become 4,1,3,2,5,6,8,7,9,10 then batch by batch
        # batch 1: 4,1
        # batch 2: 3,2 and so on
        num_workers=NUM_WORKERS,
        # number of cpu cores availabe where multiple sample can be loaded
        # let's say 4 cores, then 4 samples can be loaded at a time
        # worker 1 first sample, worker 2 second sample, worker 3 third sample, worker 4 fourth sample
        # and so on
        pin_memory=True,
        # pin memory is used to speed up the transfer of data to the GPU
        # it will copy the data to the pinned memory before transferring to the GPU
    )

    gen = Generator().to(DEVICE)
    # create the generator model and move it to the device (GPU or CPU)
    disc = Discriminator().to(DEVICE)
    # create the discriminator model and move it to the device (GPU or CPU)

    # Load pre-trained generator weights
    gen.load_state_dict(torch.load(PRETRAINED_GEN_PATH, map_location=DEVICE))
    # add the pre-trained generator weights to the generator model
    # we pretrained first and saved the weights

    opt_gen = optim.Adam(gen.parameters(), lr=LEARNING_RATE_GEN, betas=(0.9, 0.999))
    # create the optimizer for the generator
    # what is this adam? adaptive moment estimation
    # adjust the weight to reduce the loss
    # my words:
    # when one loop done, generator will backpropagate to update weight and this weight are weight that optimize loss
    # what is betas?
    # 0.9 is proposed by the paper
    # 0.999 is default in Pytorch
    # why do we use it and not other values?
    # higher beta1 means remember previous gradients more (previous direction)
    #  optimizer to build up more momentum, which helps it to be more stable, 
    # converge faster in consistent directions, and potentially escape suboptimal local minima by "carrying" itself past them.

    # what is learning rate?
    # magnitude of adjustment made to the weight in each iteration
    # if we adjust too much we might go over the optimal point
    # if we adjust too little we might take too long to reach the optimal point
    opt_disc = optim.Adam(disc.parameters(), lr=LEARNING_RATE_DISC, betas=(0.9, 0.999))

    perceptual_loss_fn = PerceptualLoss(DEVICE, LAMBDA_VGG, LAMBDA_ADV)
    # call the perceptual loss function
    # the weight is used from the paper 1 and 0.001
    bce_loss = nn.BCEWithLogitsLoss()
    #The formula for BCEWithLogitsLoss(input, target) is roughly: `target * log(sigmoid(input)) - (1 - target) * log(1 - sigmoid(input))`
    # Plugging these into the BCEWithLogitsLoss formula (and considering that target is 1):
    # Note that `torch.ones_like(disc_fake_output)` is the target, filled with ones
    #`Loss = -1 * log(sigmoid(disc_fake_output)) - (1 - 1) * log(1 - sigmoid(disc_fake_output))`
    #`Loss = -log(sigmoid(disc_fake_output))`

    # torch.ones_lik will create tensor of similar shape but filled with ones

    print("--- Starting SRGAN Training ---")
    for epoch in range(NUM_EPOCHS_GAN):
        gen.train()
        # set the generator to training mode
        disc.train()
        # set the discriminator to training mode
        loop = tqdm(loader, leave=True)

        for lr, hr in loop:
            lr = lr.to(DEVICE)
            hr = hr.to(DEVICE)

            # --- Train Discriminator ---
            gen_hr = gen(lr)

            disc_real_out = disc(hr)
            # using the hr images 
            # tensor([[[[-0.1948]]]] get something like this
            disc_fake_out = disc(gen_hr.detach())
            # sth like this tensor([[[[-0.1948]]]])
            # why detech?
            # stop graident going to generator
            # we are training the discriminator so we only update the discriminator weights

            disc_loss_real = bce_loss(disc_real_out, torch.ones_like(disc_real_out))
            #The formula for BCEWithLogitsLoss(input, target) is roughly: `target * log(sigmoid(input)) - (1 - target) * log(1 - sigmoid(input))`
            # Plugging these into the BCEWithLogitsLoss formula (and considering that target is 1):
            # Note that `torch.ones_like(disc_fake_output)` is the target, filled with ones
            #`Loss = -1 * log(sigmoid(disc_fake_output)) - (1 - 1) * log(1 - sigmoid(disc_fake_output))`
            #`Loss = -log(sigmoid(disc_fake_output))`

            # even real images go through the discriminator, they will be some loss
            # because discriminator is learning to distinguish real and fake images
            disc_loss_fake = bce_loss(disc_fake_out, torch.zeros_like(disc_fake_out))

            disc_loss = (disc_loss_real + disc_loss_fake) / 2
            # Average the losses for real and fake images
            # why average?
            # because we want to balance the loss for real and fake images
            # why 
            # ideal nash equilibrium is when both generator and discriminator have equal loss
            # so after average (x + x )/2 = x we cannot tell real or fake beacuase generator output
            # is undistinguishable from real images
            # give equal importance to both real and fake images
            # linear law the two variable has linear relationship wrt disc_loss
            # that means both increase or decrease together

            opt_disc.zero_grad()
            disc_loss.backward()
            opt_disc.step()

            # --- Train Generator ---
            # why do this?
            # in the paper: We alternate updates to the generator
            # and discriminator network
            disc_fake_for_gen = disc(gen_hr)
            # why do we need this?
            # we get output from the discriminator
            # so in perceptual loss we can calculate the adversarial loss
            gen_loss = perceptual_loss_fn(disc_fake_for_gen, gen_hr, hr)
            # this one just supply variables to calculate the perceptual loss

            opt_gen.zero_grad()
            gen_loss.backward()
            opt_gen.step()

            loop.set_postfix(g_loss=gen_loss.item(), d_loss=disc_loss.item())

        print(f"Epoch [{epoch+1}/{NUM_EPOCHS_GAN}]")
        torch.save(gen.state_dict(), GEN_PATH)
        # save the generator model
        torch.save(disc.state_dict(), DISC_PATH)

    print("--- Finished SRGAN Training ---")